# Graph Neural Networks using PyTorch Geometric

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snss
import pandas as pd
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Check available hardware
import torch
if torch.cuda.is_available():
    print(f'GPU is available, using GPU {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
elif torch.mps.is_available():
    print(f'Apple Metal Performance Shaders framework is available, using {torch.device("mps")}')
    device = torch.device('mps')
else:
    print(f'Hardware acceleration not available, using CPU')
    device = torch.device('cpu')

In [ ]:
!pip install torch torchvision torchaudio torch-geometric networkx

## Dataset: [QM9 Molecular benchmark dataset](https://pytorch-geometric.readthedocs.io/en/2.6.1/generated/torch_geometric.datasets.QM9.html)

In [ ]:
import os
from torch_geometric.datasets import QM9

dataset = QM9(root=os.path.expanduser('~/scr10'))
print(f'Dataset has length {len(dataset)}')

In [ ]:
data = dataset[0]
data

| Attribute | Shape | What is it? | 
| - | - | - |
| x | `[num_atoms, num_features]` | node feature matrix, atomic properties |
| edge_index | `[2, num_edges]` | connectivity matrix (atomic bonds) |
| edge_attr| `[num_edges, num_features]` | edge feature matrix, bond properties |
| y | `[1, num_targets]` | targets, molecular properties |
| z | `[num_atoms,]` | atomic numbers of each atom |
| pos | `[num_atoms, 3]` | cartesian coordinates of each atom |
| idx | `[1]` | index of the molecule |
| name | `string` | name of the molecule |

In [ ]:
print(data.x)

In [ ]:
import networkx as nx
from torch_geometric.utils import to_networkx

G = to_networkx(data, node_attrs=['x', 'z'], edge_attrs=None)
G.nodes(data=True)

In [ ]:
element_map = {1: 'H', 6: 'C'}
nx.get_node_attributes(G, 'z')
elements = {key: element_map[value] for key, value in nx.get_node_attributes(G, 'z').items()}
elements

In [ ]:
nx.draw(
    G,
    labels=elements
)

In [ ]:
# Send data to GPU
data.to(device)
data.is_cuda

## Graph convolutions with torch-geometric

In [ ]:
import torch_geometric
torch_geometric.nn.GCNConv?

In [ ]:
import torch.nn as tnn
import torch_geometric.nn as gnn
import torch.nn.functional as F

class BasicGNN(tnn.Module):
    def __init__(self, f_node):
        super().__init__()

    def forward(self, data):
        x = data.x # Shape [N, F_in]
        edge_index = data.edge_index # [2, E], adjacency information
        batch_vector = data.batch # Shape [N,], maps each node to a graph (batch_matrix)

model = BasicGNN(data.x.shape[1]).to(device)
print(model(data))
model

## Training the model

In [ ]:
from torch.utils.data import random_split

train_dataset, val_dataset, test_dataset = random_split(dataset, [0.6, 0.2, 0.2])
print(f'Train dataset has length {len(train_dataset)}')
print(f'Validation dataset has length {len(val_dataset)}')
print(f'Test dataset has length {len(test_dataset)}')

In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

batch = next(iter(train_loader))
batch

In [ ]:
# Batch vector

In [ ]:
# Batch offset for each graph


In [ ]:
# First graph in batch


In [ ]:
# Pooling operations


In [ ]:
model(batch.to(device))

In [ ]:
from tqdm.notebook import tqdm

def train(model, optimizer, train_loader, val_loader, num_epochs=10, target_idx=1):
    train_loss = []
    val_loss = []
    
    for epoch in range(num_epochs):
    
        loss_ = 0
        graphs_ = 0
        model.train()
        for batch in tqdm(train_loader, leave=False):
            # Send data to device
            batch.to(device)
    
            # Make a prediction, compute the loss
            y_pred = model(batch)
            loss = F.mse_loss(y_pred, batch.y[:, target_idx, None])
    
            # Backprop, apply gradients, reset optimizer
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    
            # Logging
            loss_ += loss.item()
            graphs_ += batch.num_graphs
    
        train_loss.append(loss_ / graphs_)
        
        loss_ = 0
        graphs_ = 0
        model.eval()
        with torch.no_grad():
            for batch in tqdm(val_loader, leave=False):
                # Send data to device
                batch.to(device)
        
                # Make a prediction, compute the loss
                y_pred = model(batch)
                loss = F.mse_loss(y_pred, batch.y[:, target_idx, None])
        
                # Logging
                loss_ += loss.item()
                graphs_ += batch.num_graphs
    
        val_loss.append(loss_ / graphs_)
        print(f"Epoch: {epoch} | "
              f"train loss: {train_loss[-1]:.3f} | "
              f"val loss: {val_loss[-1]:.3f}")

    return train_loss, val_loss

model = BasicGNN(data.x.shape[1])
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

train_loss, val_loss = train(model, optimizer, train_loader, val_loader, num_epochs=5)

In [ ]:
fig, ax = plt.subplots(1, 1, sharex=True, figsize=(3, 2.5))

ax.plot(train_loss, color='black', label='Train')
ax.plot(val_loss, color='red', label='Validation')
ax.set(xlabel='Epoch', ylabel='Loss')

fig.legend(*ax.get_legend_handles_labels(), 
           loc='center left', bbox_to_anchor=[1, 0.5], framealpha=0)

plt.tight_layout()

In [ ]:
y_true, y_pred = [], []
model.eval()
target_idx = 1

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
with torch.no_grad():
    for batch in tqdm(test_loader):
        yp = model(batch.to(device))
        
        y_true.append(batch.y[:, target_idx].cpu().numpy())
        y_pred.append(yp.squeeze().cpu().numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
print(y_true.shape, y_pred.shape)

In [ ]:
from sklearn.metrics import r2_score
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(y_true, y_pred, s=5, alpha=0.5, label=f'Test ($R^2 = {r2_score(y_true, y_pred):.2f}$)')
ax.plot([0,100], [0, 100], color='red', linestyle='--')
ax.set(
    xlabel='True Isotropic Polarizability [$a_0^3$]',
    ylabel='Predicted Isotropic Polarizability [$a_0^3$]',
    aspect='equal',
)
ax.legend();

print(f'Mean absolute difference: {np.mean(np.abs(y_true - y_pred)):.1f} a_0^3')

## Scaled Dot Product Attention

Attention allows the model to determine *importance weights* for each incoming message. That is, 
\begin{align*}
    \mathbf{x}^{(t+1)}_i = \mathbf{W}_1 \mathbf{x}^{(t)}_i + \sum_{j \in \mathcal{N}(i)} \alpha_{ij} \ \mathbf{W}_V \mathbf{x}_j^{(t)}
\end{align*}

Here $W_1$ is the self-update matrix, $W_V$ is a matrix that maps neighbor node features into an incoming message *value*, and $\alpha_{ij}$ is an **attention weight** set by the alignment between a node's *query* and a neighbor's *key*.  


\begin{align*}
    \alpha_{ij} = \text{softmax} \left( 
        \frac{
            \left( \mathbf{W}_Q \mathbf{x}_i \right)^{\top} \left( \mathbf{W}_K \mathbf{x}_j \right) 
        }{\sqrt{d}}
    \right)
\end{align*}

In [ ]:
import torch_geometric
torch_geometric.nn.TransformerConv?

In [ ]:
import torch.nn as tnn
import torch_geometric.nn as gnn
import torch.nn.functional as F

class ScaledDotProductAttentionGNN(tnn.Module):
    def __init__(self, f_node, f_edge):
        super().__init__()

    def forward(self, data):
        x = data.x # Shape [N, F_in]
        edge_index = data.edge_index # [2, E]
        batch_vector = data.batch # Shape [N,], maps each node to a graph

model = ScaledDotProductAttentionGNN(data.x.shape[1], data.edge_attr.shape[1]).to(device)
print(model(data))
model

In [ ]:
model = ScaledDotProductAttentionGNN(data.x.shape[1], data.edge_attr.shape[1])
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

train_loss, val_loss = train(model, optimizer, train_loader, val_loader, num_epochs=5)

In [ ]:
fig, ax = plt.subplots(1, 1, sharex=True, figsize=(3, 2.5))

ax.plot(train_loss, color='black', label='Train')
ax.plot(val_loss, color='red', label='Validation')
ax.set(xlabel='Epoch', ylabel='Loss')

fig.legend(*ax.get_legend_handles_labels(), 
           loc='center left', bbox_to_anchor=[1, 0.5], framealpha=0)

plt.tight_layout()

In [ ]:
y_true, y_pred = [], []
model.eval()
target_idx = 1

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
with torch.no_grad():
    for batch in tqdm(test_loader):
        yp = model(batch.to(device))
        
        y_true.append(batch.y[:, target_idx].cpu().numpy())
        y_pred.append(yp.squeeze().cpu().numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
print(y_true.shape, y_pred.shape)

In [ ]:
from sklearn.metrics import r2_score
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(y_true, y_pred, s=5, alpha=0.5, label=f'Test ($R^2 = {r2_score(y_true, y_pred):.2f}$)')
ax.plot([0,100], [0, 100], color='red', linestyle='--')
ax.set(
    xlabel='True Isotropic Polarizability [$a_0^3$]',
    ylabel='Predicted Isotropic Polarizability [$a_0^3$]',
    aspect='equal',
)
ax.legend();

print(f'Mean absolute difference: {np.mean(np.abs(y_true - y_pred)):.1f} a_0^3')

## Graph Attention Networks

As another attention mechanism, let's consider the Graph Attention Network formulation ([link](https://arxiv.org/abs/1710.10903)). Here, the updated value at each node is a weighted aggregation
\begin{align*}
    \mathbf{x}^{(t+1)}_i = \sum_{j \in \mathcal{N}(i) \cup \{i\}} \alpha_{ij} \mathbf{W} \mathbf{x}_j
\end{align*}

The weights are computed using a normalized *attention* mechanism

\begin{align*}
    \alpha_{ij} = \frac{
        \exp \left( \text{LeakyReLU} \left( 
            \mathbf{a}_s \mathbf{W} \mathbf{x}^{(t)}_i + 
            \mathbf{a}_t \mathbf{W} \mathbf{x}^{(t)}_j + 
            \mathbf{a}_e \mathbf{W}_e \mathbf{e}_{ij} \right) \right) 
    }{
        \sum_{k \in \mathcal{N}(i) \cup \{i\}} \exp \left( \text{LeakyReLU} \left( 
            \mathbf{a}_s \mathbf{W} \mathbf{x}^{(t)}_i + 
            \mathbf{a}_t \mathbf{W} \mathbf{x}^{(t)}_k + 
            \mathbf{a}_e \mathbf{W}_e \mathbf{e}_{ik} \right) \right) 
    }
\end{align*}

Here, $\mathbf{W}$ is a learnable linear transformation that is applied to all node values. $\mathbf{a}_s, \mathbf{a}_t$ are weight vectors applied to the transformed feature values. There is an additional transformation matrix and weight for edge features in our case. The denominator is a normalization factor meant to ensure that all incoming weights to a particular node sum to 1.

This looks like a complete mess, right? Well really, what this is doing is applying a single-layer MLP to the concatenated node and neighbor features. 
\begin{align*}
    \tilde{\alpha}_{ij} &= \text{LeakyRELU} \left( \left[ \mathbf{a}_s, \mathbf{a}_n \right] \cdot \left[ \mathbf{W}\, \mathbf{x}_i^{(t)}, \mathbf{W} \,\mathbf{x}_{j}^{(t)} \right] \right) \\
    &= \text{LeakyRELU} \left( \mathbf{a} \cdot \mathbf{h} \right) \\
    \alpha_{ij} &= \text{softmax}_j ( \tilde{\alpha}_{ij})
\end{align*}

In [ ]:
import torch_geometric
torch_geometric.nn.GATConv?

In [ ]:
import torch.nn as tnn
import torch_geometric.nn as gnn
import torch.nn.functional as F

class AttentionGNN(tnn.Module):
    def __init__(self, f_node, f_edge):
        super().__init__()

    def forward(self, data):
        x = data.x # Shape [N, F_in]
        edge_index = data.edge_index # [2, E]
        batch_vector = data.batch # Shape [N,], maps each node to a graph

model = AttentionGNN(data.x.shape[1], data.edge_attr.shape[1]).to(device)
print(model(data))
model

In [ ]:
model = AttentionGNN(data.x.shape[1], data.edge_attr.shape[1])
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

train_loss, val_loss = train(model, optimizer, train_loader, val_loader, num_epochs=5)

In [ ]:
fig, ax = plt.subplots(1, 1, sharex=True, figsize=(3, 2.5))

ax.plot(train_loss, color='black', label='Train')
ax.plot(val_loss, color='red', label='Validation')
ax.set(xlabel='Epoch', ylabel='Loss')

fig.legend(*ax.get_legend_handles_labels(), 
           loc='center left', bbox_to_anchor=[1, 0.5], framealpha=0)

plt.tight_layout()

In [ ]:
y_true, y_pred = [], []
model.eval()

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
with torch.no_grad():
    for batch in tqdm(test_loader):
        yp = model(batch.to(device))
        
        y_true.append(batch.y[:, target_idx].cpu().numpy())
        y_pred.append(yp.squeeze().cpu().numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
print(y_true.shape, y_pred.shape)

In [ ]:
from sklearn.metrics import r2_score
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(y_true, y_pred, s=5, alpha=0.5, label=f'Test ($R^2 = {r2_score(y_true, y_pred):.2f}$)')
ax.plot([0,100], [0, 100], color='red', linestyle='--')
ax.set(
    xlabel='True Isotropic Polarizability [$a_0^3$]',
    ylabel='Predicted Isotropic Polarizability [$a_0^3$]',
    aspect='equal',
)
ax.legend();

print(f'Mean absolute difference: {np.mean(np.abs(y_true - y_pred)):.1f} a_0^3')

## Edge-aware convolutions with torch-geometric

Our edges contain important features too! For example, they could indicate whether there is an ionic or covalent bond, etc. So our message passing needs to accommodate these edge features. 

`NNConv` is a layer that learns an **edge-conditioned convolution** 
\begin{align*}
    \mathbf{x}^{(t+1)}_i = \mathbf{x}^{(t)}_i \mathbf{W} + \sum_{j \in \mathcal{N}(i)} \mathbf{x}^{(t)}_j h_{\theta}(\mathbf{e}_{ij})
\end{align*}
where $h_{\theta}$ can be a neural network such as an MLP to represent an arbitrary functional relationship between the edge features and the corresponding importance weight.

Is this a permutation invariant operation?

In [ ]:
import torch_geometric
torch_geometric.nn.NNConv?

In [ ]:
import torch.nn as tnn
import torch_geometric.nn as gnn
import torch.nn.functional as F

class EdgeAwareGNN(tnn.Module):
    def __init__(self, f_node, f_edge):
        super().__init__()

    def forward(self, data):
        x = data.x # Shape [N, F_in]
        edge_index = data.edge_index # [2, E]
        batch_vector = data.batch # Shape [N,], maps each node to a graph

model = EdgeAwareGNN(data.x.shape[1], data.edge_attr.shape[1]).to(device)
print(model(data))
model

In [ ]:
model = EdgeAwareGNN(data.x.shape[1], data.edge_attr.shape[1])
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

train_loss, val_loss = train(model, optimizer, train_loader, val_loader, num_epochs=5)

In [ ]:
fig, ax = plt.subplots(1, 1, sharex=True, figsize=(3, 2.5))

ax.plot(train_loss, color='black', label='Train')
ax.plot(val_loss, color='red', label='Validation')
ax.set(xlabel='Epoch', ylabel='Loss')

fig.legend(*ax.get_legend_handles_labels(), 
           loc='center left', bbox_to_anchor=[1, 0.5], framealpha=0)

plt.tight_layout()

In [ ]:
y_true, y_pred = [], []
model.eval()

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
with torch.no_grad():
    for batch in tqdm(test_loader):
        yp = model(batch.to(device))
        
        y_true.append(batch.y[:, target_idx].cpu().numpy())
        y_pred.append(yp.squeeze().cpu().numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
print(y_true.shape, y_pred.shape)

In [ ]:
from sklearn.metrics import r2_score
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(y_true, y_pred, s=5, alpha=0.5, label=f'Test ($R^2 = {r2_score(y_true, y_pred):.2f}$)')
ax.plot([0,100], [0, 100], color='red', linestyle='--')
ax.set(
    xlabel='True Isotropic Polarizability [$a_0^3$]',
    ylabel='Predicted Isotropic Polarizability [$a_0^3$]',
    aspect='equal',
)
ax.legend();

print(f'Mean absolute difference: {np.mean(np.abs(y_true - y_pred)):.1f} a_0^3')

## A veritable zoo of graph operations

You can get a big list of GNN operations [here](https://pytorch-geometric.readthedocs.io/en/2.6.1/modules/nn.html#convolutional-layers). Pick one and implement it. What score do you get?